## 1. Libraries

In [ ]:
# Libraries
import os
import time
import random
import logging
import gc
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import neuralforecast

from pathlib import Path
from neuralforecast import NeuralForecast
from neuralforecast.models import KAN, NHITS, NBEATSx, GRU, LSTM, MLP
from neuralforecast.losses.pytorch import MAE

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

## 2. Configuration

In [ ]:
base_path = Path.cwd()
dataset_files = {
    "HUPA": "HUPA_UCM_IOB_COB.parquet",
    "AZT1D": "AZT1D_IOB_COB.parquet"
}
datasets = list(dataset_files)
RESUME = True
USE_AZT1D_BASAL = False  # Enable only after confirming causal availability
run_name = "local_ablation_18_v1"
freq = "5min"
horizons = [6, 12, 18, 24]
test_size = 288
input_size = 36
max_steps = 200
random_seed = 42
warmup_hours = 8
stat_exog_list = []

np.random.seed(random_seed)
random.seed(random_seed)
torch.manual_seed(random_seed)
torch.cuda.manual_seed_all(random_seed)
print("GPU:", torch.cuda.get_device_name(0))
output_path = base_path / "results_ablation" / run_name
output_path.mkdir(parents=True, exist_ok=True)

## 3. Ablation experiments

In [ ]:
insulin_cols = ["basal_rate", "bolus_volume_delivered"]
iob_cols = [f"IOB_DIA_{dia}h" for dia in range(2, 9)]
activity_cols = ["heart_rate", "steps", "calories"]

experiments = {
    "E01": [],
    "E02": insulin_cols,
    "E03": ["carb_input"],
    "E04": ["COB"],
    "E05": ["IOB_DIA_2h"],
    "E06": ["IOB_DIA_3h"],
    "E07": ["IOB_DIA_4h"],
    "E08": ["IOB_DIA_5h"],
    "E09": ["IOB_DIA_6h"],
    "E10": ["IOB_DIA_7h"],
    "E11": ["IOB_DIA_8h"],
    "E12": iob_cols,
    "E13": insulin_cols + ["carb_input"],
    "E14": insulin_cols + ["COB"],
    "E15": iob_cols + ["carb_input"],
    "E16": iob_cols + ["COB"],
    "E17": insulin_cols + ["COB"] + activity_cols,
    "E18": iob_cols + ["COB"] + activity_cols
}

def get_experiments(data, dataset):
    available = {
        name: features for name, features in experiments.items()
        if all(c in data.columns and np.isfinite(data[c].to_numpy(dtype=float)).any()
               for c in features)
    }
    if dataset == "AZT1D" and not USE_AZT1D_BASAL:
        available = {name: features for name, features in available.items()
                     if "basal_rate" not in features}
    return available

## 4. Data preparation

In [ ]:
def load_dataset(dataset):
    data = pd.read_parquet(base_path / dataset_files[dataset])
    data = data.rename(columns={"time": "ds", "glucose": "y"})
    data["ds"] = pd.to_datetime(data["ds"])
    data = data.sort_values(["unique_id", "ds"]).reset_index(drop=True)
    if data[["unique_id", "ds"]].isna().any().any() or data.duplicated(["unique_id", "ds"]).any():
        raise ValueError("Missing or duplicate patient timestamps")
    return data


def prepare_patient(data, patient, common_features, dataset):
    s = (data.loc[data["unique_id"] == patient]
         .sort_values("ds").reset_index(drop=True).copy())
    valid = pd.Series(np.isfinite(s[["y"] + common_features]
                                  .to_numpy(dtype=float)).all(axis=1), index=s.index)
    # Gaps must never be joined as if they were consecutive.
    breaks = s["ds"].diff().ne(pd.Timedelta(freq))
    if dataset == "AZT1D":
        # Proxy for incomplete event history, using the final Parquet only.
        event_cols = ["bolus_volume_delivered", "carb_input"]
        unknown_event = ~pd.Series(np.isfinite(s[event_cols].to_numpy(dtype=float))
                                   .all(axis=1), index=s.index)
        last_reset = s["ds"].where(breaks | unknown_event).ffill()
        valid &= (s["ds"] - last_reset).ge(pd.Timedelta(hours=warmup_hours))
        valid &= ~unknown_event
    block_id = (breaks | ~valid).cumsum()
    min_required = test_size + input_size + max(horizons)
    blocks = [block for _, block in s.loc[valid].groupby(block_id[valid], sort=True)
              if len(block) >= min_required]
    if not blocks:
        raise ValueError(f"No continuous common block with >= {min_required} samples")
    # Most recent eligible block, common to all experiments and horizons.
    return blocks[-1].reset_index(drop=True).copy()

## 5. Model configuration

In [ ]:
def create_models(horizon, input_size, hist_exog_list):
    common_kwargs = dict(
        h=horizon,
        input_size=input_size,
        futr_exog_list=[],
        hist_exog_list=hist_exog_list,
        stat_exog_list=stat_exog_list,
        loss=MAE(),
        max_steps=max_steps,
        learning_rate=1e-3,
        scaler_type="standard",
        random_seed=random_seed,
        batch_size=16,
        valid_batch_size=16,
        start_padding_enabled=True,
        val_check_steps=50,
        early_stop_patience_steps=-1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        accelerator="gpu",
        devices=1,
        enable_checkpointing=False
    )
    models = [
        KAN(**common_kwargs, hidden_size=512, n_hidden_layers=1, alias="KAN"),
        NHITS(**common_kwargs, n_blocks=[1, 1, 1],
              mlp_units=[[512, 512], [512, 512], [512, 512]], alias="NHITS"),
        NBEATSx(**common_kwargs, stack_types=["identity", "trend", "seasonality"],
                n_blocks=[1, 1, 1], mlp_units=[[512, 512], [512, 512], [512, 512]],
                alias="NBEATSx"),
        GRU(**common_kwargs, encoder_n_layers=2, encoder_hidden_size=200,
            decoder_hidden_size=128, decoder_layers=2, alias="GRU"),
        LSTM(**common_kwargs, encoder_n_layers=2, encoder_hidden_size=200,
             decoder_hidden_size=128, decoder_layers=2, alias="LSTM"),
        MLP(**common_kwargs, hidden_size=512, num_layers=2, alias="MLP")
    ]
    return models

## 6. Prediction format

In [ ]:
def format_neuralforecast_forecasts(df_, sample_type, model_cols):
    df_ = df_.copy()
    df_["ds"] = pd.to_datetime(df_["ds"])
    df_["cutoff"] = pd.to_datetime(df_["cutoff"])
    out = df_.melt(id_vars=["unique_id", "ds", "cutoff", "y"],
                   value_vars=model_cols, var_name="model", value_name="forecast")
    out["forecast_raw"] = out["forecast"]
    out["forecast"] = out["forecast"].clip(lower=0)
    out["type"] = sample_type
    out["frequency"] = freq
    out["horizon"] = ((out["ds"] - out["cutoff"]) / pd.Timedelta(freq)).astype(int)
    return out

## 7. Local walk-forward training

In [ ]:
def train_local(data_patient, dataset, patient, experiment, hist_exog_list, horizon):
    data_nf = data_patient[["unique_id", "ds", "y"] + hist_exog_list].copy()
    assert data_nf["unique_id"].nunique() == 1
    step_size = horizon
    n_windows = test_size // horizon

    # New independent model instances for this patient/experiment/horizon.
    models = create_models(horizon, input_size, hist_exog_list)
    model_names = [m.alias for m in models]
    nf = NeuralForecast(models=models, freq=freq)

    t0 = time.perf_counter()
    cv_raw = nf.cross_validation(
        df=data_nf, n_windows=n_windows, step_size=step_size, val_size=0,
        refit=True, verbose=False, id_col="unique_id", time_col="ds", target_col="y"
    )
    cv_time = time.perf_counter() - t0
    cv_raw["ds"] = pd.to_datetime(cv_raw["ds"])
    cv_raw["cutoff"] = pd.to_datetime(cv_raw["cutoff"])
    cv_out = format_neuralforecast_forecasts(cv_raw, "out_sample", model_names)
    if not np.isfinite(cv_out[["y", "forecast_raw"]].to_numpy()).all():
        raise ValueError("Non-finite predictions")

    pred_all = cv_out.assign(
        dataset=dataset, experiment=experiment, training_type="local",
        cv_time_seconds=cv_time, horizon_config=horizon, n_windows=n_windows,
        step_size=step_size, input_size=input_size, max_steps=max_steps,
        features=",".join(["CGM"] + hist_exog_list), static_features="",
        train_start=data_nf["ds"].min(), initial_cutoff=cv_raw["cutoff"].min(),
        test_end=cv_raw["ds"].max()
    )
    pred_all["prediction_minutes"] = pred_all["horizon"] * 5
    pred_all["horizon_minutes"] = horizon * 5
    print(f"CV completed in {cv_time:.2f} seconds")
    return pred_all

## 8. Automatic training

In [ ]:
def atomic_parquet(frame, file):
    temp = file.with_suffix(".tmp.parquet")
    frame.to_parquet(temp, index=False)
    os.replace(temp, file)

config = dict(datasets=datasets, experiments=experiments, horizons=horizons,
              freq=freq, test_size=test_size, input_size=input_size,
              max_steps=max_steps, random_seed=random_seed, warmup_hours=warmup_hours,
              refit=True, scaler="standard", loss="MAE",
              use_azt1d_basal=USE_AZT1D_BASAL,
              neuralforecast=neuralforecast.__version__, torch=str(torch.__version__))
config_file = output_path / "configuration.json"
if config_file.exists() and json.loads(config_file.read_text()) != config:
    raise ValueError("Configuration changed; use a new run_name")
config_file.write_text(json.dumps(config, indent=2), encoding="utf-8")

patient_audit = []
log = []
for dataset in datasets:
    data = load_dataset(dataset)
    available_experiments = get_experiments(data, dataset)
    common_features = sorted({c for features in available_experiments.values()
                              for c in features})
    print(dataset, "| Available experiments:", list(available_experiments))
    print(dataset, "| Not available:", sorted(set(experiments) - set(available_experiments)))
    for patient in sorted(data["unique_id"].unique()):
        total = int(data["unique_id"].eq(patient).sum())
        try:
            data_patient = prepare_patient(data, patient, common_features, dataset)
        except ValueError as e:
            patient_audit.append(dict(dataset=dataset, patient=patient, eligible=False,
                                      total_samples=total, reason=str(e)))
            pd.DataFrame(patient_audit).to_csv(output_path / "patient_audit.csv", index=False)
            continue

        patient_audit.append(dict(dataset=dataset, patient=patient, eligible=True,
                                  total_samples=total, selected_samples=len(data_patient),
                                  train_start=data_patient["ds"].min(),
                                  initial_cutoff=data_patient["ds"].iloc[-test_size-1],
                                  test_end=data_patient["ds"].max()))
        pd.DataFrame(patient_audit).to_csv(output_path / "patient_audit.csv", index=False)

        for experiment, hist_exog_list in available_experiments.items():
            for horizon in horizons:
                folder = output_path / dataset / experiment / f"H{horizon * 5}" / str(patient)
                folder.mkdir(parents=True, exist_ok=True)
                file = folder / "predictions.parquet"
                meta = dict(dataset=dataset, patient=patient, experiment=experiment,
                            horizon_minutes=horizon * 5)
                if RESUME and file.exists():
                    log.append(dict(**meta, status="skipped", error=""))
                    continue
                print("Training:", meta)
                try:
                    pred_all = train_local(data_patient, dataset, patient, experiment,
                                           hist_exog_list, horizon)
                    atomic_parquet(pred_all, file)
                    log.append(dict(**meta, status="completed", error=""))
                    del pred_all
                except Exception as e:
                    log.append(dict(**meta, status="failed", error=repr(e)))
                    print("Training error:", repr(e))
                finally:
                    pd.DataFrame(log).to_csv(output_path / "training_log.csv", index=False)
                    gc.collect()
                    torch.cuda.empty_cache()
pd.DataFrame(log).to_csv(output_path / "training_log.csv", index=False)
print("Training finished. Review training_log.csv before comparing results.")

## 9. Consolidated predictions and metrics

In [ ]:
prediction_files = sorted(output_path.rglob("predictions.parquet"))
if not prediction_files:
    raise ValueError("No predictions saved. Check training_log.csv")

pred_all = pd.concat([pd.read_parquet(file) for file in prediction_files], ignore_index=True)
out_df = pred_all.query("type == 'out_sample'").copy()
out_df["absolute_error"] = np.abs(out_df["y"] - out_df["forecast"])
out_df["squared_error"] = (out_df["y"] - out_df["forecast"]) ** 2
keys = ["dataset", "unique_id", "experiment", "model", "horizon_minutes"]

def calculate_metrics(group_cols):
    metrics = (out_df.groupby(group_cols)
               .agg(MAE=("absolute_error", "mean"), MSE=("squared_error", "mean"),
                    samples=("y", "size")).reset_index())
    metrics["RMSE"] = np.sqrt(metrics.pop("MSE"))
    return metrics

metrics_horizon = calculate_metrics(keys + ["prediction_minutes"])
metrics = calculate_metrics(keys)
metrics_final = metrics_horizon.loc[metrics_horizon["prediction_minutes"].eq(
    metrics_horizon["horizon_minutes"])].copy()

atomic_parquet(pred_all, output_path / "predictions_all.parquet")
metrics.to_csv(output_path / "metrics.csv", index=False)
metrics_horizon.to_csv(output_path / "metrics_horizon.csv", index=False)
metrics_final.to_csv(output_path / "metrics_final.csv", index=False)
print("Prediction files:", len(prediction_files))
display(metrics.head(12).round(3))
# Compare only matched patients / test periods; see patient_audit.csv and training_log.csv.

## 10. Plot a patient

In [ ]:
# Select any available combination; defaults to the first completed one.
selected = out_df[["dataset", "unique_id", "experiment", "horizon_minutes"]].iloc[0]
s = out_df.loc[np.logical_and.reduce([out_df[c].eq(selected[c]).to_numpy()
                                      for c in selected.index])].copy()
model_colors = {"NHITS": "blue", "NBEATSx": "red", "KAN": "green", "GRU": "purple",
                "LSTM": "orange", "MLP": "brown"}
fig, ax = plt.subplots(figsize=(16, 5))
actual = s.groupby("ds", as_index=False)["y"].first().sort_values("ds")
ax.plot(actual["ds"], actual["y"], color="black", linewidth=2, label="Actual")
for model, g in s.sort_values("ds").groupby("model", sort=False):
    ax.plot(g["ds"], g["forecast"], color=model_colors.get(model, "gray"),
            linestyle="--", linewidth=1.5, label=model)
ax.axhline(70, color="red", linestyle=":")
ax.axhline(180, color="orange", linestyle=":")
ax.set_title(f"{selected['unique_id']} | {selected['experiment']} | {selected['horizon_minutes']} min")
ax.set_ylabel("Glucose (mg/dL)")
ax.set_xlabel("Time")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.grid(True, alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(ncol=7, frameon=False)
plt.tight_layout()
plt.show()